In [ ]:
from openai import OpenAI
import pandas as pd
import time



# Changed: Instantiate the OpenAI client directly with the API key
client = OpenAI(api_key="")

MODEL = "gpt-4o-mini"


# ---------------------------------------
# LLM CALL FUNCTION
# ---------------------------------------

def ask_llm(prompt, temperature=0.3):


    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )


    return response.choices[0].message.content


# ---------------------------------------
# BASELINE
# ---------------------------------------

def baseline_answer(question):

    prompt = f"""
Answer the question.

Question:
{question}

Give the final answer.
"""

    return ask_llm(prompt, temperature=0)


# ---------------------------------------
# STATIC CHAIN OF THOUGHT
# ---------------------------------------

def static_chain_answer(question):

    prompt = f"""
Solve the problem step-by-step.

Question:
{question}

Explain reasoning clearly and give the final answer.
"""

    return ask_llm(prompt)


# ---------------------------------------
# APC INITIAL REASONING
# ---------------------------------------

def apc_initial_answer(question):

    prompt = f"""
Solve the problem step-by-step.

Verify your reasoning before giving the final answer.

Question:
{question}

End with:

Final Answer: <answer>
"""

    return ask_llm(prompt)


# ---------------------------------------
# CONFIDENCE CHECK
# ---------------------------------------

def confidence_check(answer):

    prompt = f"""
Evaluate the confidence of the following reasoning.

Reasoning:
{answer}

Is this reasoning reliable?

Respond with ONE word:

HIGH
MEDIUM
LOW
"""

    return ask_llm(prompt, temperature=0)


# ---------------------------------------
# CRITIC-JUDGE VERIFIER
# ---------------------------------------

def verify_answer(question, answer):

    prompt = f"""
You are a strict reasoning critic.

Question:
{question}

Answer:
{answer}

Step 1:
Identify any possible problems:
- logical errors
- unsupported claims
- hallucinated facts
- missing reasoning steps

Step 2:
Explain briefly if a mistake exists.

Step 3:
Give the final verdict.

If ANY issue exists write:

VERDICT: INCORRECT

If reasoning is completely valid write:

VERDICT: CORRECT
"""

    result = ask_llm(prompt, temperature=0)

    if "INCORRECT" in result:
        return "INCORRECT"
    else:
        return "CORRECT"


# ---------------------------------------
# REFINEMENT
# ---------------------------------------

def refine_answer(question, answer):

    prompt = f"""
The previous reasoning may contain mistakes.

Question:
{question}

Previous reasoning:
{answer}

Tasks:

1 Identify the mistake
2 Correct the reasoning
3 Solve the problem again

Write step-by-step reasoning.

End with:

Final Answer: <answer>
"""

    return ask_llm(prompt)


# ---------------------------------------
# LOAD DATASET
# ---------------------------------------

df = pd.read_csv("/content/merged_all_questions.csv")

results = []

refinement_count = 0


# ---------------------------------------
# MAIN LOOP
# ---------------------------------------

for i, row in df.iterrows():

    question = row["question"]
    dataset = row["task"]
    ground_truth = row["ground_truth"]

    print(f"\nProcessing Question {i+1}")

    # ------------------
    # BASELINE
    # ------------------

    baseline = baseline_answer(question)

    # ------------------
    # STATIC CHAIN
    # ------------------

    static_chain = static_chain_answer(question)

    # ------------------
    # APC INITIAL
    # ------------------

    apc_initial = apc_initial_answer(question)

    confidence = confidence_check(apc_initial)

    verification = verify_answer(question, apc_initial)

    apc_final = apc_initial

    refinement_triggered = False

    # ------------------
    # REFINEMENT TRIGGER
    # ------------------

    if confidence in ["LOW", "MEDIUM"] or verification == "INCORRECT":

        refinement_triggered = True
        refinement_count += 1

        apc_refined = refine_answer(question, apc_initial)

        verification2 = verify_answer(question, apc_refined)

        apc_final = apc_refined

        # ------------------
        # SECOND REFINEMENT
        # ------------------

        if verification2 == "INCORRECT":

            apc_final = refine_answer(question, apc_refined)

    # ------------------
    # SAVE RESULT
    # ------------------

    results.append({

        "dataset": dataset,
        "question": question,
        "ground_truth": ground_truth,

        "baseline": baseline,
        "static_chain": static_chain,

        "apc_initial": apc_initial,
        "apc_final": apc_final,

        "refinement_triggered": refinement_triggered

    })

    time.sleep(1)


# ---------------------------------------
# SAVE CSV
# ---------------------------------------

results_df = pd.DataFrame(results)

results_df.to_csv("apc_results.csv", index=False)

print("\nExperiment Finished")

print(f"Total Refinements Triggered: {refinement_count}")